In [0]:
import requests

# Generate token using your ArcGIS credentials
token_url = "https://www.arcgis.com/sharing/rest/generateToken"

params = {
    "username": "csuguest10",
    "password": "CSUguest10!",
    "referer": "https://services1.arcgis.com/KNdRU5cN6ENqCTjk/arcgis/rest/services/PPQ%20Fruit%20Fly%20Detections%20Summary%20Feature%20Layer/FeatureServer",
    "f": "json"
}

response = requests.post(token_url, data=params)
token_data = response.json()
if 'error' in token_data:
    raise Exception(f"Error generating token: {token_data['error']['message']}")

token = token_data['token']

print("Token generated successfully!")



Token generated successfully!


In [0]:

from arcgis.gis import GIS
import requests
import pandas as pd
from arcgis.features import GeoAccessor, GeoSeriesAccessor


from arcgis.gis import GIS
from arcgis.features import FeatureLayer

gis = GIS("https://www.arcgis.com", "csuguest10", "CSUguest10!", "https://services1.arcgis.com/KNdRU5cN6ENqCTjk/arcgis/rest/services/PPQ%20Fruit%20Fly%20Detections%20Summary%20Feature%20Layer/FeatureServer")

groups = gis.groups.search('title:2026 Hackathon Problem statement 1 Fruit Flies', max_groups=5)

target_group = groups[0]
group_items = target_group.content()


feature_items = [item for item in group_items if item.type == 'Feature Service']

target_item = feature_items[1]

target_item = feature_items[1]
feature_layer = target_item.layers[0]
#for layer in feature_layers:    
    #for field in layer.properties.fields:
        #print(f"{field['name']} ({field['type']}): {field['alias']}")

print("Layer name:", feature_layer.properties.name)
print("\nAvailable fields:")
for field in feature_layer.properties.fields:
    print(f"  - {field['name']} ({field['type']})")

a = feature_layer.query(
    where="1=1",
    out_fields="STATE_NAME, OBJECTID, NAME, monthyear, CommonName",
    return_geometry=True
)

print(f"Number of features: {len(a.features)}")
df = a.sdf

display(df.head(20))

Layer name: Fruit Fly Detections

Available fields:
  - OBJECTID (esriFieldTypeOID)
  - NAME (esriFieldTypeString)
  - STATE_NAME (esriFieldTypeString)
  - monthyear (esriFieldTypeString)
  - CommonName (esriFieldTypeString)
  - Count_ (esriFieldTypeDouble)
  - Shape__Area (esriFieldTypeDouble)
  - Shape__Length (esriFieldTypeDouble)
Number of features: 305


,OBJECTID,STATE_NAME,NAME,monthyear,CommonName,SHAPE
0,1,Texas,Cameron County,01/2025,Mexican Fruit Fly,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
1,2,Texas,Cameron County,02/2022,Mexican Fruit Fly,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
2,3,Texas,Cameron County,02/2024,Mexican Fruit Fly,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
3,4,Texas,Cameron County,02/2025,Mexican Fruit Fly,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
4,5,Texas,Cameron County,03/2022,Mexican Fruit Fly,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
5,6,Texas,Cameron County,03/2024,Mexican Fruit Fly,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
6,7,Texas,Cameron County,03/2025,Mexican Fruit Fly,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
7,8,Texas,Cameron County,04/2022,Mexican Fruit Fly,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
8,9,Texas,Cameron County,04/2024,Mexican Fruit Fly,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."
9,10,Texas,Cameron County,04/2025,Mexican Fruit Fly,"{""rings"": [[[-119012.8452, -1626525.7954], [-1..."


In [0]:
from arcgis.gis import GIS

gis = GIS("https://www.arcgis.com", "csuguest10", "CSUguest10!")

# Get items
groups = gis.groups.search('title:2026 Hackathon Problem statement 1 Fruit Flies', max_groups=5)
target_group = groups[0]
group_items = target_group.content()
feature_items = [item for item in group_items if item.type == 'Feature Service']
target_item = feature_items[1]

# Create map
map1 = gis.map("United States")
map1.content.add(target_item)

# Export to HTML file (doesn't load in notebook)
map1.export_to_html("fruit_fly_map.html", title="Fruit Fly Detections")
print("Map exported to fruit_fly_map.html")

Map exported to fruit_fly_map.html


In [0]:
# Query data
feature_set = feature_layer.query(
    where="1=1",
    out_fields="*",
    return_geometry=True
)

df = feature_set.sdf

# Check what geometry types you have
print("Checking geometry types...")
geom_types = df['SHAPE'].apply(lambda x: type(x).__name__).value_counts()
print(geom_types)

# Extract coordinates based on geometry type
def smart_coordinate_extract(geom):
    """Handle different geometry types"""
    if geom is None:
        return None, None
    
    geom_type = type(geom).__name__
    
    try:
        if geom_type == 'Point':
            return geom.y, geom.x
        elif geom_type in ['Polygon', 'Polyline', 'MultiPoint']:
            # Use centroid for complex geometries
            centroid = geom.centroid
            return centroid[1], centroid[0]
        else:
            print(f"Unknown geometry type: {geom_type}")
            return None, None
    except Exception as e:
        print(f"Error with {geom_type}: {e}")
        return None, None


# Export
# Convert directly to GeoJSON and save
geojson = feature_set.to_geojson
with open("fruit_fly_data.geojson", "w") as f:
    f.write(geojson)

print("Exported as GeoJSON!")

Checking geometry types...
SHAPE
Polygon    305
Name: count, dtype: int64
Exported as GeoJSON!


BTS T-100 Preliminary

In [0]:
import requests
import pandas as pd

url = "https://data.bts.gov/resource/3xj5-daif.json"

# Optional: add limit (default is small)
params = {
    "$limit": 50000
}

response = requests.get(url, params=params)
data = response.json()

df = pd.DataFrame(data)
df.head()

,year,month,date,checks,market,segment,market_fit,segment_fit
0,2019,1,2019-01-01T00:00:00.000,61694899,76927270,77736885,76033251.05,76652855.74
1,2019,2,2019-02-01T00:00:00.000,58535547,72133079,72709473,72046017.62,72637909.71
2,2019,3,2019-03-01T00:00:00.000,72714771,90766871,91465187,89940787.14,90657053.87
3,2019,4,2019-04-01T00:00:00.000,69754997,87177374,88008446,86205429.44,86895734.19
4,2019,5,2019-05-01T00:00:00.000,74582398,92577085,93462082,92297809.97,93030458.8


In [0]:
spark_df = spark.createDataFrame(df)
display(spark_df)

year,month,date,checks,market,segment,market_fit,segment_fit
2019,1,2019-01-01T00:00:00.000,61694899,76927270,77736885,76033251.05,76652855.74
2019,2,2019-02-01T00:00:00.000,58535547,72133079,72709473,72046017.62,72637909.71
2019,3,2019-03-01T00:00:00.000,72714771,90766871,91465187,89940787.14,90657053.87
2019,4,2019-04-01T00:00:00.000,69754997,87177374,88008446,86205429.44,86895734.19
2019,5,2019-05-01T00:00:00.000,74582398,92577085,93462082,92297809.97,93030458.8
2019,6,2019-06-01T00:00:00.000,76489355,95766379,96693780,94704468.89,95453844.82
2019,7,2019-07-01T00:00:00.000,78841453,99421298,100354651,97672914.25,98442921.86
2019,8,2019-08-01T00:00:00.000,76083824,96224458,97072964,94192671.93,94938490.69
2019,9,2019-09-01T00:00:00.000,66622674,83142668,83892858,82252307.73,82915136.96
2019,10,2019-10-01T00:00:00.000,72020967,89246813,89913019,89065177.68,89775358.63


BTS T100 

In [0]:


import requests
import pandas as pd

url = "https://data.bts.gov/resource/56rv-9p75.json"

# Optional: add limit (default is small)
params = {
    "$limit": 50000
}

response = requests.get(url, params=params)
data = response.json()

df = pd.DataFrame(data)
df.head()


,country,year,inbound_international,inbound_international_1,inbound_international_seats,inbound_international_load,inbound_international_2,inbound_international_seats_1,inbound_international_distance,inbound_international_distance_1,inbound_international_payload,inbound_international_freight,inbound_international_mail,outbound_international,outbound_international_1,outbound_international_seats,outbound_international_load,outbound_international_2,outbound_international_seats_1,outbound_international_3,outbound_international_4,outbound_international_payload,outbound_international_freight,outbound_international_mail,country_name
0,AE,2014,6031,1648936,2007135,82.2,273.4100480848947,332.803017741668,7396.877632233461,7398.066874639161,759052802,117833749,5842,6396,1620545,2014673,80.4,253.3685115697311,314.9895247029393,7408.243902439024,7410.875221607545,842294532,172928367,1594395,United Arab Emirates
1,AE,2015,7824,2162146,2737439,79,276.3479038854805,349.877172801636,7426.763675869121,7431.601353470117,996319438,153112644,306912,8209,2046420,2733784,74.9,249.289803873797,333.0227798757462,7438.07199415276,7428.423166309946,1088411911,227863094,1793300,United Arab Emirates
2,AE,2016,8091,2279596,2907772,78.4,281.7446545544432,359.3835125448028,7435.822518848103,7409.450182400741,1068958592,135251436,22068,8518,2139679,2909200,73.5,251.1949988260155,341.5355717304532,7445.209673632308,7416.397495605649,1161827186,216748173,122045,United Arab Emirates
3,AE,2017,7545,2121193,2776965,76.4,281.138899933731,368.0536779324055,7464.854340622929,7417.283688471534,972297916,141017101,0,7926,1984247,2775101,71.5,250.346580873076,350.1262932122129,7470.399823366137,7417.965352599752,1061653603,192492029,797,United Arab Emirates
4,AE,2018,6429,2045403,2500758,81.8,318.1525898273449,388.9808679421372,7414.685176543786,7399.557104394587,861577320,115494328,0,7034,1940169,2522912,76.9,275.8272675575775,358.6738697753767,7415.688512937162,7401.619115138938,991524269,208755244,321283,United Arab Emirates


USDA

In [0]:


url = "https://api.fas.usda.gov/api/gats/commodities"

params = {
    "api_key": "DRNqyyv97g0Zs3XeDfVnDTrabO1CG7oaAKgQewdz",
    "commodityCode": "0803",  # example: fruit category
    "year": "2023"
}

response = requests.get(url, params=params)
data = response.json()

df = pd.DataFrame(data)
df.head()

,hS10Code,startDate,endDtate,productType,commodityName,commodityDescription,isAgCommodity,censusUOMId1,censusUOMId2,fasConvertedUOMId,fasNonConvertedUOMId
0,0011010,196701,None,B,"CTL,DAIRY,BREDNG",None,False,78,78,78,78
1,0011020,196701,None,B,"CTL,BF,BRG,X BUL",None,False,78,78,78,78
2,0011030,196701,None,B,"BLLS,BF,BREEDING",None,False,78,78,78,78
3,0011040,196701,None,B,"CTL,X BREEDING",None,False,78,78,78,78
4,0012000,196701,None,B,"SHEEP,LAMBS,GOAT",None,False,78,78,78,78


## Fruit Fly CSV → ArcGIS Map (Semi-transparent circles by species & severity)

In [0]:
%pip install arcgis --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
databricks-connect 18.0.2 requires pandas<3,>=1.0.5, but you have pandas 3.0.2 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
from arcgis.gis import GIS
from arcgis.geocoding import geocode
import time

# Read the CSV
csv_path = "/Workspace/hackathon-root/hackathon-team10/fruit_fly_data.csv"
df_csv = pd.read_csv(csv_path)
print(f"Loaded {len(df_csv)} rows")
print(f"Species: {df_csv['Common name'].unique()}")
print(f"Severity range: {df_csv['Severity'].min()} - {df_csv['Severity'].max()}")
display(df_csv.head(10))

Loaded 41 rows
Species: ['Oriental fruit fly' 'Mediterranean fruit fly' 'Mexican fruit fly'
 'European cherry fruit fly' 'Tau fruit fly' 'Queensland fruit fly'
 'Sapote fruit fly']
Severity range: 1 - 5


Location,Date,Common name,Arrival,Severity
"San Bernardino & Riverside Counties, CA",January 2024,Oriental fruit fly,Plane,4
California,March 2024,Oriental fruit fly,Plane,4
California,March 2024,Mediterranean fruit fly,Plane,4
Texas,April 2024,Mexican fruit fly,Plane,3
Texas,April 2024,Mexican fruit fly,Plane,4
"Santa Clara County, CA",May 2024,Oriental fruit fly,Plane,1
"Cayuga, Genesee & Ontario Counties, NY",June 2024,European cherry fruit fly,Plane,4
Texas,June 2024,Mexican fruit fly,Plane,4
"Sacramento County, CA",June 2024,Oriental fruit fly,Plane,1
"Los Angeles County, CA",July 2024,Tau fruit fly,Plane,1


In [0]:
# Connect to CSU RAMS ArcGIS portal
gis_csu = GIS("https://csurams.maps.arcgis.com", "csuguest10", "CSUguest10!")
print(f"Connected as: {gis_csu.properties.user.username}")
print(f"Organization: {gis_csu.properties.name}")

Connected as: csuguest10
Organization: Colorado State University


In [0]:
# Geocode each unique location to avoid redundant API calls
unique_locations = df_csv['Location'].unique()
print(f"Geocoding {len(unique_locations)} unique locations...")

location_coords = {}
for loc in unique_locations:
    try:
        results = geocode(loc, as_featureset=False)
        if results:
            lat = results[0]['location']['y']
            lon = results[0]['location']['x']
            location_coords[loc] = (lat, lon)
            print(f"  ✓ {loc} → ({lat:.4f}, {lon:.4f})")
        else:
            print(f"  ✗ {loc} → No results")
            location_coords[loc] = (None, None)
    except Exception as e:
        print(f"  ✗ {loc} → Error: {e}")
        location_coords[loc] = (None, None)
    time.sleep(0.3)  # rate limiting

# Apply coordinates to dataframe
df_csv['latitude'] = df_csv['Location'].map(lambda x: location_coords.get(x, (None, None))[0])
df_csv['longitude'] = df_csv['Location'].map(lambda x: location_coords.get(x, (None, None))[1])

# Check for failures
failed = df_csv[df_csv['latitude'].isna()]
if len(failed) > 0:
    print(f"\nWARNING: {len(failed)} rows failed geocoding")
else:
    print(f"\nAll {len(df_csv)} rows geocoded successfully!")

display(df_csv.head())

Geocoding 16 unique locations...
  ✓ San Bernardino & Riverside Counties, CA → (38.0293, -121.9619)
  ✓ California → (36.3741, -119.2702)
  ✓ Texas → (31.4627, -99.3331)
  ✓ Santa Clara County, CA → (37.2325, -121.6962)
  ✓ Cayuga, Genesee & Ontario Counties, NY → (43.2209, -77.2833)
  ✓ Sacramento County, CA → (38.4495, -121.3441)
  ✓ Los Angeles County, CA → (34.1731, -118.2571)
  ✓ Cameron County, TX → (26.1514, -97.4528)
  ✓ Contra Costa & Riverside Counties, CA → (37.9526, -122.3310)
  ✓ Riverside & San Bernardino Counties, CA → (38.0293, -121.9619)
  ✓ Hidalgo County, TX → (26.3967, -98.1811)
  ✓ Alameda & Santa Clara Counties, CA → (37.3319, -121.9032)
  ✓ Riverside County, CA → (33.7437, -115.9937)
  ✓ Cameron, Hidalgo & Willacy Counties, TX → (25.9813, -97.5158)
  ✓ Hidalgo & Starr Counties, TX → (26.0894, -97.9603)
  ✓ Orange County, CA → (33.6769, -117.7768)

All 41 rows geocoded successfully!


Location,Date,Common name,Arrival,Severity,latitude,longitude
"San Bernardino & Riverside Counties, CA",January 2024,Oriental fruit fly,Plane,4,38.029295982807,-121.961912960728
California,March 2024,Oriental fruit fly,Plane,4,36.374105693,-119.27023
California,March 2024,Mediterranean fruit fly,Plane,4,36.374105693,-119.27023
Texas,April 2024,Mexican fruit fly,Plane,3,31.462733046,-99.33305009
Texas,April 2024,Mexican fruit fly,Plane,4,31.462733046,-99.33305009


In [0]:
from arcgis.features import FeatureSet, Feature
from arcgis.geometry import Point
import json

# Build point features from geocoded data
features = []
for _, row in df_csv.iterrows():
    pt = Point({"x": row['longitude'], "y": row['latitude'], 
                "spatialReference": {"wkid": 4326}})
    attr = {
        "Location": row['Location'],
        "Date": row['Date'],
        "CommonName": row['Common name'],
        "Arrival": row['Arrival'],
        "Severity": int(row['Severity'])
    }
    features.append(Feature(geometry=pt, attributes=attr))

print(f"Created {len(features)} point features")

# Define the feature layer schema
fields = [
    {"name": "Location", "type": "esriFieldTypeString", "alias": "Location", "length": 200},
    {"name": "Date", "type": "esriFieldTypeString", "alias": "Date", "length": 50},
    {"name": "CommonName", "type": "esriFieldTypeString", "alias": "Species", "length": 100},
    {"name": "Arrival", "type": "esriFieldTypeString", "alias": "Arrival Method", "length": 50},
    {"name": "Severity", "type": "esriFieldTypeInteger", "alias": "Severity (1-5)"}
]

# Define species-to-color mapping (semi-transparent RGBA)
species_colors = {
    "Oriental fruit fly":        [230, 25, 75, 140],    # Red
    "Mediterranean fruit fly":   [60, 180, 75, 140],    # Green
    "Mexican fruit fly":         [0, 130, 200, 140],    # Blue
    "European cherry fruit fly": [245, 130, 48, 140],   # Orange
    "Tau fruit fly":             [145, 30, 180, 140],   # Purple
    "Queensland fruit fly":      [240, 50, 230, 140],   # Magenta
    "Sapote fruit fly":          [210, 180, 40, 140],   # Yellow-gold
}

# Build unique value infos for the renderer
# Circle size scales with severity: sev 1 = 8px, sev 5 = 40px
min_size = 8
max_size = 40

unique_value_infos = []
for species, color in species_colors.items():
    unique_value_infos.append({
        "value": species,
        "label": species,
        "symbol": {
            "type": "esriSMS",
            "style": "esriSMSCircle",
            "color": color,
            "size": 18,  # default size, overridden by visual variable
            "outline": {
                "color": [50, 50, 50, 200],
                "width": 1
            }
        }
    })

# Renderer: unique values by species + proportional size by severity
renderer = {
    "type": "uniqueValue",
    "field1": "CommonName",
    "uniqueValueInfos": unique_value_infos,
    "defaultSymbol": {
        "type": "esriSMS",
        "style": "esriSMSCircle",
        "color": [128, 128, 128, 140],
        "size": 12,
        "outline": {"color": [50, 50, 50, 200], "width": 1}
    },
    "defaultLabel": "Other",
    "visualVariables": [
        {
            "type": "sizeInfo",
            "field": "Severity",
            "minDataValue": 1,
            "maxDataValue": 5,
            "minSize": min_size,
            "maxSize": max_size
        }
    ]
}

# Build drawing info
drawing_info = {"renderer": renderer}

print("Renderer configured:")
print(f"  - {len(unique_value_infos)} species colors (semi-transparent)")
print(f"  - Circle size: {min_size}px (severity 1) to {max_size}px (severity 5)")

Created 41 point features
Renderer configured:
  - 7 species colors (semi-transparent)
  - Circle size: 8px (severity 1) to 40px (severity 5)


In [0]:
import json
import os

# Create a GeoJSON FeatureCollection for publishing
geojson_features = []
for _, row in df_csv.iterrows():
    geojson_features.append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [row['longitude'], row['latitude']]
        },
        "properties": {
            "Location": row['Location'],
            "Date": row['Date'],
            "CommonName": row['Common name'],
            "Arrival": row['Arrival'],
            "Severity": int(row['Severity'])
        }
    })

geojson_collection = {
    "type": "FeatureCollection",
    "features": geojson_features
}

# Save to file for upload
geojson_path = "/tmp/fruit_fly_circles.geojson"
with open(geojson_path, "w") as f:
    json.dump(geojson_collection, f)

print(f"GeoJSON saved: {len(geojson_features)} features")

# Publish to CSU RAMS ArcGIS as a hosted feature layer
item_props = {
    "title": "Fruit Fly Detections - Circle Map",
    "tags": "fruit fly, hackathon, detections, severity",
    "description": "Fruit fly detection data with semi-transparent circles colored by species and sized by severity (1-5).",
    "type": "GeoJson"
}

# Check if item already exists and delete it to avoid duplicates
existing = gis_csu.content.search(query="title:'Fruit Fly Detections - Circle Map' owner:csuguest10", max_items=10)
for item in existing:
    print(f"Deleting existing item: {item.title} ({item.id})")
    item.delete()

# Upload GeoJSON
geojson_item = gis_csu.content.add(item_props, data=geojson_path)
print(f"Uploaded GeoJSON item: {geojson_item.title} (ID: {geojson_item.id})")

# Publish as hosted feature layer
published_item = geojson_item.publish()
print(f"Published feature layer: {published_item.title} (ID: {published_item.id})")
print(f"URL: {published_item.url}")

GeoJSON saved: 41 features


/databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3577: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Uploaded GeoJSON item: Fruit Fly Detections - Circle Map (ID: 566559b6cfc94f1faa4063181f153e30)
Published feature layer: Fruit Fly Detections - Circle Map (ID: 1e171d9c151d4ca5b8c1f9e295da789d)
URL: https://services1.arcgis.com/KNdRU5cN6ENqCTjk/arcgis/rest/services/Fruit_Fly_Detections_Circle_Map/FeatureServer


In [0]:
from arcgis.features import FeatureLayer

# Get the feature layer and apply the renderer
fl = published_item.layers[0]

# Update the layer's drawing info with our custom renderer
update_result = fl.manager.update_definition({
    "drawingInfo": drawing_info
})
print(f"Renderer update result: {update_result}")

# Share the layer publicly so it can be viewed in Map Viewer
published_item.share(everyone=True)
print("Layer shared publicly")

# Generate the Map Viewer URL
map_viewer_url = f"https://csurams.maps.arcgis.com/apps/mapviewer/index.html?layers={published_item.id}"
print(f"\n{'='*60}")
print(f"Open your map here:")
print(f"{map_viewer_url}")
print(f"{'='*60}")
print(f"\nLayer details:")
print(f"  - Item ID: {published_item.id}")
print(f"  - Title: {published_item.title}")
print(f"  - Features: {fl.query(return_count_only=True)}")
print(f"  - Species colors: 7 unique (semi-transparent)")
print(f"  - Size range: {min_size}px (severity 1) to {max_size}px (severity 5)")

Renderer update result: {'success': True}


/databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3577: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Layer shared publicly

Open your map here:
https://csurams.maps.arcgis.com/apps/mapviewer/index.html?layers=1e171d9c151d4ca5b8c1f9e295da789d

Layer details:
  - Item ID: 1e171d9c151d4ca5b8c1f9e295da789d
  - Title: Fruit Fly Detections - Circle Map
  - Features: 41
  - Species colors: 7 unique (semi-transparent)
  - Size range: 8px (severity 1) to 40px (severity 5)


## Add Historical Detections (records_11) — Grey circles, no species

In [0]:
import pandas as pd
from arcgis.geocoding import geocode
import time

# records_11 from Cut Cell Backup — historical detections (has month+year, no species)
records_11 = [
    ("Rancho Cucamonga, CA", "March", 2012, 2),
    ("Miami, FL", "June", 2015, 4),
    ("Philadelphia, PA", "November", 2016, 2),
    ("Dulles Airport, VA", "August", 2019, 2),
    ("Brownsville, TX", "February", 2020, 4),
    ("Upland, CA", "July", 2021, 4),
    ("North Hills, CA", "April", 2022, 4),
    ("Fountain Valley, CA", "June", 2022, 4),
    ("McAllen, TX", "July", 2022, 2),
    ("Brownsville, TX", "July", 2022, 2),
    ("JFK Airport, NY", "December", 2022, 2),
    ("JFK Airport, NY", "January", 2023, 2),
    ("Los Angeles, CA", "November", 2023, 5),
    ("Anaheim, CA", "November", 2023, 5),
    ("San Jose, CA", "November", 2023, 5),
    ("Martinez, CA", "November", 2023, 5),
    ("Sacramento, CA", "November", 2023, 5),
    ("San Bernardino, CA", "November", 2023, 5),
    ("Riverside, CA", "November", 2023, 5),
    ("San Diego, CA", "November", 2023, 5),
    ("Ventura, CA", "November", 2023, 5),
    ("California", "November", 2023, 5),
    ("Fremont, CA", "January", 2024, 4),
    ("McAllen, TX", "January", 2024, 4),
    ("Zapata, TX", "January", 2024, 2),
    ("Fremont, CA", "March", 2024, 4),
    ("Riverside, CA", "June", 2024, 4),
    ("Martinez, CA", "June", 2024, 4),
    ("McAllen, TX", "June", 2024, 4),
    ("Brownsville, TX", "June", 2024, 2),
    ("Los Angeles, CA", "August", 2024, 4),
    ("San Bernardino, CA", "August", 2024, 4),
    ("McAllen, TX", "August", 2024, 2),
    ("Fremont, CA", "September", 2024, 5),
    ("Riverside, CA", "September", 2024, 4),
    ("San Bernardino, CA", "September", 2024, 4),
    ("McAllen, TX", "October", 2024, 4),
    ("Brownsville, TX", "October", 2024, 2),
    ("Zapata, TX", "October", 2024, 2),
    ("Fremont, CA", "January", 2025, 5),
    ("San Jose, CA", "January", 2025, 4),
    ("McAllen, TX", "January", 2025, 4),
    ("Detroit, MI", "June", 2025, 2),
    ("Riverside, CA", "July", 2025, 2),
    ("Martinez, CA", "July", 2025, 1),
    ("Fremont, CA", "September", 2025, 5),
    ("Riverside, CA", "September", 2025, 4),
    ("San Bernardino, CA", "September", 2025, 4)
]

df_hist = pd.DataFrame(records_11, columns=["Location", "Month", "Year", "Severity"])
print(f"Historical detections: {len(df_hist)} rows")
print(f"Unique locations: {df_hist['Location'].nunique()}")

# Geocode unique locations
unique_locs = df_hist['Location'].unique()
print(f"\nGeocoding {len(unique_locs)} unique locations...")

hist_coords = {}
for loc in unique_locs:
    try:
        results = geocode(loc, as_featureset=False)
        if results:
            lat = results[0]['location']['y']
            lon = results[0]['location']['x']
            hist_coords[loc] = (lat, lon)
            print(f"  ✓ {loc} → ({lat:.4f}, {lon:.4f})")
        else:
            print(f"  ✗ {loc} → No results")
            hist_coords[loc] = (None, None)
    except Exception as e:
        print(f"  ✗ {loc} → Error: {e}")
        hist_coords[loc] = (None, None)
    time.sleep(0.3)

df_hist['latitude'] = df_hist['Location'].map(lambda x: hist_coords.get(x, (None, None))[0])
df_hist['longitude'] = df_hist['Location'].map(lambda x: hist_coords.get(x, (None, None))[1])

failed = df_hist[df_hist['latitude'].isna()]
if len(failed) > 0:
    print(f"\nWARNING: {len(failed)} rows failed geocoding")
    display(failed)
else:
    print(f"\nAll {len(df_hist)} rows geocoded successfully!")

display(df_hist.head(10))

Historical detections: 48 rows
Unique locations: 23

Geocoding 23 unique locations...
  ✓ Rancho Cucamonga, CA → (34.1064, -117.5758)
  ✓ Miami, FL → (25.7751, -80.1947)
  ✓ Philadelphia, PA → (39.9511, -75.1656)
  ✓ Dulles Airport, VA → (38.9536, -77.4521)
  ✓ Brownsville, TX → (25.9030, -97.4989)
  ✓ Upland, CA → (34.0997, -117.6503)
  ✓ North Hills, CA → (34.2356, -118.4763)
  ✓ Fountain Valley, CA → (33.7107, -117.9457)
  ✓ McAllen, TX → (26.1967, -98.2357)
  ✓ JFK Airport, NY → (40.6398, -73.7787)
  ✓ Los Angeles, CA → (34.0522, -118.2433)
  ✓ Anaheim, CA → (33.8345, -117.9156)
  ✓ San Jose, CA → (37.3348, -121.8881)
  ✓ Martinez, CA → (38.0190, -122.1349)
  ✓ Sacramento, CA → (38.5769, -121.4950)
  ✓ San Bernardino, CA → (34.1083, -117.2941)
  ✓ Riverside, CA → (33.9805, -117.3770)
  ✓ San Diego, CA → (32.7158, -117.1638)
  ✓ Ventura, CA → (34.2808, -119.2931)
  ✓ California → (36.3741, -119.2702)
  ✓ Fremont, CA → (37.5520, -121.9781)
  ✓ Zapata, TX → (26.9004, -99.2675)
  ✓ Det

Location,Month,Year,Severity,latitude,longitude
"Rancho Cucamonga, CA",March,2012,2,34.1063986,-117.575764
"Miami, FL",June,2015,4,25.775084,-80.194702
"Philadelphia, PA",November,2016,2,39.9510605,-75.1656204
"Dulles Airport, VA",August,2019,2,38.953617149847,-77.452096251169
"Brownsville, TX",February,2020,4,25.903046,-97.498884
"Upland, CA",July,2021,4,34.0997359,-117.6502985
"North Hills, CA",April,2022,4,34.235589,-118.47633
"Fountain Valley, CA",June,2022,4,33.710735,-117.945728
"McAllen, TX",July,2022,2,26.196695,-98.235698
"Brownsville, TX",July,2022,2,25.903046,-97.498884


In [0]:
from arcgis.gis import GIS
from arcgis.features import Feature, FeatureLayer
from arcgis.geometry import Point

# Connect to CSU RAMS ArcGIS
gis_csu = GIS("https://csurams.maps.arcgis.com", "csuguest10", "CSUguest10!")
print(f"Connected as: {gis_csu.properties.user.username}")

# Get the existing published feature layer
published_item = gis_csu.content.get("1e171d9c151d4ca5b8c1f9e295da789d")
fl = published_item.layers[0]
print(f"Existing layer: {published_item.title}")
print(f"Layer URL: {fl.url}")

# Build features for historical detections
# CommonName = "Unknown (Historical)" so the grey default symbol is used
new_features = []
for _, row in df_hist.iterrows():
    if pd.isna(row['latitude']) or pd.isna(row['longitude']):
        continue
    pt = Point({"x": row['longitude'], "y": row['latitude'],
                "spatialReference": {"wkid": 4326}})
    attr = {
        "Location": row['Location'],
        "Date": f"{row['Month']} {int(row['Year'])}",
        "CommonName": "Unknown (Historical)",
        "Arrival": "Plane",
        "Severity": int(row['Severity'])
    }
    new_features.append(Feature(geometry=pt, attributes=attr))

print(f"\nAppending {len(new_features)} historical features (grey circles)...")

# Append to the existing feature layer
result = fl.edit_features(adds=new_features)
add_count = sum(1 for r in result['addResults'] if r['success'])
fail_count = sum(1 for r in result['addResults'] if not r['success'])
print(f"Added: {add_count}, Failed: {fail_count}")

# Try to get new count
try:
    count = fl.query(return_count_only=True)
    print(f"New total feature count: {count}")
except:
    print(f"Features appended (count query unavailable)")

Connected as: csuguest10
Existing layer: Fruit Fly Detections - Circle Map
Layer URL: https://services1.arcgis.com/KNdRU5cN6ENqCTjk/arcgis/rest/services/Fruit_Fly_Detections_Circle_Map/FeatureServer/0

Appending 48 historical features (grey circles)...
Added: 48, Failed: 0
Features appended (count query unavailable)


In [0]:
# Update the renderer to include an explicit entry for grey "Unknown (Historical)"
# and keep the existing species colors
species_colors = {
    "Oriental fruit fly":        [230, 25, 75, 140],
    "Mediterranean fruit fly":   [60, 180, 75, 140],
    "Mexican fruit fly":         [0, 130, 200, 140],
    "European cherry fruit fly": [245, 130, 48, 140],
    "Tau fruit fly":             [145, 30, 180, 140],
    "Queensland fruit fly":      [240, 50, 230, 140],
    "Sapote fruit fly":          [210, 180, 40, 140],
    "Unknown (Historical)":      [160, 160, 160, 140],  # Grey
}

min_size = 8
max_size = 40

unique_value_infos = []
for species, color in species_colors.items():
    unique_value_infos.append({
        "value": species,
        "label": species,
        "symbol": {
            "type": "esriSMS",
            "style": "esriSMSCircle",
            "color": color,
            "size": 18,
            "outline": {
                "color": [50, 50, 50, 200],
                "width": 1
            }
        }
    })

renderer = {
    "type": "uniqueValue",
    "field1": "CommonName",
    "uniqueValueInfos": unique_value_infos,
    "defaultSymbol": {
        "type": "esriSMS",
        "style": "esriSMSCircle",
        "color": [160, 160, 160, 140],
        "size": 12,
        "outline": {"color": [50, 50, 50, 200], "width": 1}
    },
    "defaultLabel": "Other",
    "visualVariables": [
        {
            "type": "sizeInfo",
            "field": "Severity",
            "minDataValue": 1,
            "maxDataValue": 5,
            "minSize": min_size,
            "maxSize": max_size
        }
    ]
}

drawing_info = {"renderer": renderer}

update_result = fl.manager.update_definition({"drawingInfo": drawing_info})
print(f"Renderer update: {update_result}")

map_url = f"https://csurams.maps.arcgis.com/apps/mapviewer/index.html?layers={published_item.id}"
print(f"\n{'='*60}")
print(f"Map URL (41 original + 48 historical = 89 features):")
print(f"{map_url}")
print(f"{'='*60}")
print(f"\nLegend:")
for species, color in species_colors.items():
    marker = '○' if 'Unknown' in species else '●'
    print(f"  {marker} {species}")

Renderer update: {'success': True}

Map URL (41 original + 48 historical = 89 features):
https://csurams.maps.arcgis.com/apps/mapviewer/index.html?layers=1e171d9c151d4ca5b8c1f9e295da789d

Legend:
  ● Oriental fruit fly
  ● Mediterranean fruit fly
  ● Mexican fruit fly
  ● European cherry fruit fly
  ● Tau fruit fly
  ● Queensland fruit fly
  ● Sapote fruit fly
  ○ Unknown (Historical)


## Enable Time Slider on the Map

In [0]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
from datetime import datetime
import calendar

# Connect and get the layer
gis_csu = GIS("https://csurams.maps.arcgis.com", "csuguest10", "CSUguest10!")
published_item = gis_csu.content.get("1e171d9c151d4ca5b8c1f9e295da789d")
fl = published_item.layers[0]
print(f"Layer: {published_item.title}")
print(f"URL: {fl.url}")

# Step 1: Add a Date-type field to the layer
add_field_result = fl.manager.add_to_definition({
    "fields": [{
        "name": "DetectionDate",
        "type": "esriFieldTypeDate",
        "alias": "Detection Date"
    }]
})
print(f"\nAdd date field result: {add_field_result}")

Layer: Fruit Fly Detections - Circle Map
URL: https://services1.arcgis.com/KNdRU5cN6ENqCTjk/arcgis/rest/services/Fruit_Fly_Detections_Circle_Map/FeatureServer/0

Add date field result: {'success': True}


In [0]:
import requests
import json

# Step 2: Query all features — the Date field is already epoch ms
query_url = fl.url + "/query"
params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "false",
    "f": "json",
    "token": gis_csu._con.token
}
resp = requests.get(query_url, params=params)
features_data = resp.json()

print(f"Retrieved {len(features_data['features'])} features")
sample = features_data['features'][0]['attributes']
print(f"Fields: {list(sample.keys())}")
print(f"Sample Date value: {sample.get('Date')} (type: {type(sample.get('Date')).__name__})")

# The Date field is stored as epoch ms — copy directly to DetectionDate
updates = []
for feat in features_data['features']:
    oid = feat['attributes']['ObjectId']
    date_val = feat['attributes'].get('Date')
    
    # Handle both epoch ms (int/str of digits) and text dates
    epoch_ms = None
    if date_val is not None:
        if isinstance(date_val, (int, float)):
            epoch_ms = int(date_val)
        elif isinstance(date_val, str) and date_val.strip().isdigit():
            epoch_ms = int(date_val.strip())
        else:
            # Try parsing as "Month Year"
            try:
                dt = datetime.strptime(str(date_val).strip(), "%B %Y")
                epoch_ms = int(dt.timestamp() * 1000)
            except:
                pass
    
    if epoch_ms is not None:
        updates.append({
            "attributes": {
                "ObjectId": oid,
                "DetectionDate": epoch_ms
            }
        })
    else:
        print(f"  Could not parse OID {oid}: '{date_val}'")

print(f"\nPrepared {len(updates)} updates")

# Show sample dates
for u in updates[:5]:
    epoch = u['attributes']['DetectionDate']
    dt = datetime.fromtimestamp(epoch / 1000)
    print(f"  OID {u['attributes']['ObjectId']}: {dt.strftime('%B %Y')}")

Retrieved 71 features
Fields: ['Location', 'Date', 'CommonName', 'Arrival', 'Severity', 'ObjectId', 'DetectionDate']
Sample Date value: 1704067200000 (type: int)

Prepared 71 updates
  OID 1: January 2024
  OID 6: May 2024
  OID 7: June 2024
  OID 9: June 2024
  OID 10: July 2024


In [0]:
# Step 3: Update all features with the parsed date values
# Use REST API directly for reliability
update_url = fl.url + "/applyEdits"

# Send in batches of 50
batch_size = 50
total_success = 0
total_fail = 0

for i in range(0, len(updates), batch_size):
    batch = updates[i:i+batch_size]
    edit_params = {
        "updates": json.dumps(batch),
        "f": "json",
        "token": gis_csu._con.token
    }
    resp = requests.post(update_url, data=edit_params)
    result = resp.json()
    
    if 'updateResults' in result:
        success = sum(1 for r in result['updateResults'] if r['success'])
        fail = sum(1 for r in result['updateResults'] if not r['success'])
        total_success += success
        total_fail += fail
        if fail > 0:
            for r in result['updateResults']:
                if not r['success']:
                    print(f"  Failed OID: {r}")
    else:
        print(f"  Batch error: {result}")

print(f"Updated {total_success} features, {total_fail} failures")

Updated 71 features, 0 failures


In [0]:
# Step 4: Enable time on the layer so the Map Viewer shows the time slider
# Find the actual date range from our updates
min_epoch = min(u['attributes']['DetectionDate'] for u in updates)
max_epoch = max(u['attributes']['DetectionDate'] for u in updates)

min_dt = datetime.fromtimestamp(min_epoch / 1000)
max_dt = datetime.fromtimestamp(max_epoch / 1000)
print(f"Date range: {min_dt.strftime('%B %Y')} to {max_dt.strftime('%B %Y')}")

# Enable time properties on the layer
time_info = {
    "timeInfo": {
        "startTimeField": "DetectionDate",
        "endTimeField": None,
        "timeExtent": [min_epoch, max_epoch],
        "timeInterval": 1,
        "timeIntervalUnits": "esriTimeUnitsMonths"
    }
}

result = fl.manager.update_definition(time_info)
print(f"Enable time result: {result}")

map_url = f"https://csurams.maps.arcgis.com/apps/mapviewer/index.html?layers={published_item.id}"
print(f"\n{'='*60}")
print(f"Map with time slider enabled:")
print(f"{map_url}")
print(f"{'='*60}")
print(f"\nThe time slider should appear automatically in Map Viewer.")
print(f"Drag it to filter detections by date — dots to the right")
print(f"of the slider position will be hidden.")

Date range: March 2012 to February 2026
Enable time result: {'success': True}

Map with time slider enabled:
https://csurams.maps.arcgis.com/apps/mapviewer/index.html?layers=1e171d9c151d4ca5b8c1f9e295da789d

The time slider should appear automatically in Map Viewer.
Drag it to filter detections by date — dots to the right
of the slider position will be hidden.
